In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Optional, Union
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, ClassLabel
from sklearn.metrics import accuracy_score, classification_report, f1_score

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)

In [17]:
set_seed(42)

base_dir: Path = Path("..")
data_dir: Path = base_dir / "data"
models_dir: Path = base_dir / "models"
checkpoints_dir: str = "output/checkpoints",
onnx_dir: Path = base_dir / "onnx_export"

for d in (data_dir, models_dir, onnx_dir):
    d.mkdir(parents = True, exist_ok = True)

In [ ]:
config: dict[str, Union[int, float, str, bool]] = {
    "model_name": "distilbert-base-cased",
    "max_seq_length": 512,
    "bs": 8,
    "gradient_accumulation": 2,
    "epochs": 3,
    "lr": 2e-5,
    "fp16": True,
    "min_tag_frequency": 100, # all tags with less intros are thrown away
}

# Данные

In [4]:
with open(data_dir / "arxivData.json", "r", encoding = "utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} entries")

def ParseRecord(record: dict) -> Optional[dict]:
    try:
        import ast

        title: str = record.get("title", "").strip()
        summary: str = record.get("summary", "").strip()
        tag_str: str = record.get("tag", "[]")
        tag_list: list = ast.literal_eval(tag_str) if tag_str not in ("", "[]", None) else []

        terms = [t["term"] for t in tag_list if isinstance(t, dict) and "term" in t]

        if not title or not terms:
            return None
        
        return {
            "title": title,
            "abstract": summary,
            "tags": terms,
        }
    except (ValueError, SyntaxError, KeyError) as e:
        return None

parsed: list[dict[str, str]] = [r for r in (ParseRecord(rec) for rec in raw_data) if r is not None]
df = pd.DataFrame(parsed)

del parsed, raw_data
print(f"Got {len(df)} valid intros")

Loaded 41000 entries
Got 41000 valid intros


### Предобработка тегов

In [5]:
all_tags = [tag for tags in df["tags"] for tag in tags]
tag_counter = Counter(all_tags)

valid_tags = {tag for tag, cnt in tag_counter.items() if cnt >= config["min_tag_frequency"]}
print(f"Got {len(valid_tags)} after filtering by frequency")

label2id = {tag: idx for idx, tag in enumerate(sorted(valid_tags))}
id2label = {idx: tag for tag, idx in label2id.items()}

Got 50 after filtering by frequency


In [6]:
def GetLabel(tags: list[str]) -> Optional[int]:
    for t in tags:
        if t in label2id:
            return label2id[t]
    return None

df["label"] = df["tags"].apply(GetLabel)
df = df.dropna(subset = {"label"}).copy()
df["label"] = df["label"].astype(int)

mapping_path = data_dir / "label_mapping.json"
with open(mapping_path, "w", encoding = "utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent = 4)

### Токенизация

In [11]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
print(type(tokenizer))

def TokenizeFN(examples: dict[str, list]) -> dict[str, list]:
    texts = [
        f"{t.strip()} [SEP] {a.strip()}" if a else t.strip()
        for t, a in zip(examples["title"], examples["abstract"])
    ]
    return tokenizer(texts, truncation = True, max_length = config["max_seq_length"], padding = False)

hf_dataset = Dataset.from_pandas(df[["title", "abstract", "label"]])
hf_dataset = hf_dataset.cast_column("label", ClassLabel(num_classes=len(label2id), names=list(label2id.keys())))
tokenized = hf_dataset.map(
    TokenizeFN,
    batched = True,
    remove_columns = ["title", "abstract"],
    desc = "Tokenizing",
)

train_test = tokenized.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
train_dataset = train_test["train"]

val_test = train_test["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")
val_dataset = val_test["train"]
test_dataset = val_test["test"]

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

print(f"train: {len(dataset_dict["train"])}, val: {len(dataset_dict["validation"])}, test: {len(dataset_dict["test"])}")

<class 'transformers.models.distilbert.tokenization_distilbert_fast.DistilBertTokenizerFast'>


Casting the dataset:   0%|          | 0/41000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/41000 [00:00<?, ? examples/s]

train: 32800, val: 4100, test: 4100


# Модель

In [21]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path = config["model_name"],
    num_labels = len(label2id),
    id2label = id2label,
    label2id = label2id,
    problem_type = "single_label_classification",
)

training_args: TrainingArguments = TrainingArguments(
    output_dir = str(checkpoints_dir),
    # evaluation_strategy = "epoch", 
    save_strategy = "epoch",
    learning_rate = config["lr"],
    per_device_train_batch_size = config["bs"],
    per_device_eval_batch_size = config["bs"] * 2,
    num_train_epochs = config["epochs"],
    weight_decay = 0.01,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    fp16 = True,
    save_total_limit = 1,
)

trainer: Trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = dataset_dict["train"],
    eval_dataset = dataset_dict["validation"],
    data_collator = data_collator,
    tokenizer = tokenizer,
)

train_metrics: dict[str, float] = trainer.train()
trainer.save_model(save_directory = config["output_dir"])
print(f"Training complete. Final metrics: {train_metrics}")

loading configuration file config.json from cache at /home/user/.cache/huggingface/hub/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0/config.json
Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "id2label": {
    "0": "I.2.6",
    "1": "I.2.7",
    "2": "cmp-lg",
    "3": "cond-mat.dis-nn",
    "4": "cs.AI",
    "5": "cs.CC",
    "6": "cs.CE",
    "7": "cs.CL",
    "8": "cs.CR",
    "9": "cs.CV",
    "10": "cs.CY",
    "11": "cs.DB",
    "12": "cs.DC",
    "13": "cs.DL",
    "14": "cs.DM",
    "15": "cs.DS",
    "16": "cs.GR",
    "17": "cs.GT",
    "18": "cs.HC",
    "19": "cs.IR",
    "20": "cs.IT",
    "21": "cs.LG",
    "22": "cs.LO",
    "23": "cs.MA",
    "24": "cs.MM",
    "25": "cs.NA",
    "26": "cs.NE",
    "27": "cs.NI",
    "28": "cs.PL",
    "29": "cs.RO",
    "30": "cs.SD",
    "31": "cs.SE",
   

ValueError: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.NO
- Save strategy: SaveStrategy.EPOCH